Von MP4 zu Audio + Background
Original MP4
   ↓ Extract audio (keep original SR)
   ↓ Separate vocals/background (at original SR)
   ↓ Convert vocals & background stems → 48 kHz WAV
   ↓ Denoise vocals (48 kHz WAV) for Synthesis
   ↓ Transcripe text using the original Audio
Audio + Background   



In [1]:
import os
import shutil
import shlex
import subprocess
import demucs.separate
from df.enhance import enhance, init_df, load_audio, save_audio
import json
import requests
import time

def transcribe_with_whisperx(audio_file_path, language=None, enable_diarization=True, min_speakers=1, max_speakers=10, whisperx_url="http://whisperx:5001"):
    """Transcribe audio using WhisperX API service with diarization support."""
    print(f"Transcribing audio file with WhisperX: {audio_file_path}")
    
    # Check if WhisperX service is available
    try:
        health_response = requests.get(f"{whisperx_url}/health", timeout=10)
        if health_response.status_code != 200:
            raise Exception("WhisperX service not healthy")
        print("✅ WhisperX service is healthy")
    except Exception as e:
        print(f"❌ WhisperX service unavailable: {e}")
        return None
    
    # Prepare the request
    with open(audio_file_path, 'rb') as audio_file:
        files = {'audio': audio_file}
        data = {
            'language': language or 'en',
            'diarize': str(enable_diarization).lower(),
            'min_speakers': str(min_speakers),
            'max_speakers': str(max_speakers),
            'format': 'json'
        }
        
        print(f"📤 Sending audio to WhisperX API...")
        print(f"   Language: {data['language']}")
        print(f"   Diarization: {data['diarize']}")
        print(f"   Speaker range: {min_speakers}-{max_speakers}")
        
        try:
            response = requests.post(
                f"{whisperx_url}/transcribe",
                files=files,
                data=data,
                timeout=300  # 5 minutes timeout
            )
            
            if response.status_code == 200:
                result = response.json()
                print(f"✅ Transcription complete. Found {len(result.get('segments', []))} segments.")
                
                # Print preview with speaker information if available
                for i, segment in enumerate(result.get('segments', [])[:5]):  # First 5 segments
                    start = segment.get('start', 0)
                    end = segment.get('end', 0)
                    text = segment.get('text', '').strip()
                    speaker = segment.get('speaker', None)
                    
                    if speaker:
                        print(f"[{start:06.1f} --> {end:06.1f}] [{speaker}] {text}")
                    else:
                        print(f"[{start:06.1f} --> {end:06.1f}] {text}")
                
                if len(result.get('segments', [])) > 5:
                    print("...")
                
                return result
            else:
                print(f"❌ WhisperX API error: {response.status_code}")
                print(f"Response: {response.text}")
                return None
                
        except requests.exceptions.Timeout:
            print("❌ WhisperX API timeout - file might be too large")
            return None
        except Exception as e:
            print(f"❌ Error calling WhisperX API: {e}")
            return None

def separate_vocals(input_audio_path, output_dir, model='mdx_extra'):
    """
    Separate vocals and no-vocals with Demucs and save to custom output_dir.
    """
    print(f"Separating vocals from {input_audio_path}...")

    # Run Demucs CLI separation
    cmd = f'--two-stems vocals -n {model} "{input_audio_path}"'
    demucs.separate.main(shlex.split(cmd))

    base_name = os.path.splitext(os.path.basename(input_audio_path))[0]
    demucs_output_dir = os.path.join("separated", model, base_name)

    vocals_src = os.path.join(demucs_output_dir, "vocals.wav")
    no_vocals_src = os.path.join(demucs_output_dir, "no_vocals.wav")

    os.makedirs(output_dir, exist_ok=True)

    vocals_dst = os.path.join(output_dir, f"{base_name}_vocals.wav")
    no_vocals_dst = os.path.join(output_dir, f"{base_name}_no_vocals.wav")

    shutil.copy2(vocals_src, vocals_dst)
    shutil.copy2(no_vocals_src, no_vocals_dst)

    print(f"Vocals saved to: {vocals_dst}")
    print(f"No vocals saved to: {no_vocals_dst}")

    return vocals_dst, no_vocals_dst  # Return paths of saved files

def convert_to_48k_wav(input_audio_path, output_wav_path):
    """
    Convert any audio input to 48kHz stereo WAV using ffmpeg.
    """
    print(f"Converting {input_audio_path} to 48kHz WAV at {output_wav_path}...")
    subprocess.run([
        'ffmpeg', '-y', '-i', input_audio_path,
        '-ar', '48000',
        '-ac', '2',
        output_wav_path
    ], check=True)
    print("Conversion done.")

def denoise_audio(input_wav_path, output_wav_path):
    """
    Denoise/enhance the WAV audio using DeepFilterNet.
    """
    print(f"Denoising audio {input_wav_path} to {output_wav_path}...")
    model, df_state, _ = init_df()
    audio, _ = load_audio(input_wav_path, sr=df_state.sr())
    enhanced = enhance(model, df_state, audio)
    save_audio(output_wav_path, enhanced, df_state.sr())
    print("Denoising done.")

def separate_audio_from_video(video_path, audio_out_path, muted_video_out_path):
    """Extract audio from video and save both audio and muted video separately."""
    
    # Ensure output directories exist
    os.makedirs(os.path.dirname(audio_out_path), exist_ok=True)
    os.makedirs(os.path.dirname(muted_video_out_path), exist_ok=True)
    
    # Extract audio (48 kHz mono WAV)
    audio_cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-vn", "-acodec", "pcm_s16le", "-ar", "48000", "-ac", "1",
        audio_out_path
    ]

    # Remove audio from video
    muted_video_cmd = [
        "ffmpeg", "-y", "-i", video_path,
        "-an", muted_video_out_path
    ]

    print("🔊 Extracting audio...")
    subprocess.run(audio_cmd, check=True)
    print(f"✅ Audio saved to: {audio_out_path}")

    print("🎥 Saving muted video...")
    subprocess.run(muted_video_cmd, check=True)
    print(f"✅ Muted video saved to: {muted_video_out_path}")

def write_transcription_to_json(result, output_path):
    """Write the WhisperX transcription result to a JSON file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)  # Create output dir
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON transcript saved to: {output_path}")

def write_transcription_to_srt(result, output_path):
    """Convert WhisperX result to SRT format and save."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    srt_content = ""
    for i, segment in enumerate(result.get('segments', []), 1):
        start_time = format_timestamp_srt(segment.get('start', 0))
        end_time = format_timestamp_srt(segment.get('end', 0))
        text = segment.get('text', '').strip()
        speaker = segment.get('speaker', '')
        
        if speaker:
            text = f"[{speaker}] {text}"
        
        srt_content += f"{i}\n{start_time} --> {end_time}\n{text}\n\n"
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(srt_content)
    
    print(f"✅ SRT transcript saved to: {output_path}")

def format_timestamp_srt(seconds):
    """Format timestamp for SRT format."""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = seconds % 60
    return f"{hours:02d}:{minutes:02d}:{seconds:06.3f}".replace('.', ',')


/venv/lib/python3.10/site-packages/df/io.py:9: UserWarning: `torchaudio.backend.common.AudioMetaData` has been moved to `torchaudio.AudioMetaData`. Please update the import path.
  from torchaudio.backend.common import AudioMetaData


I use a function to make sure all audio, video, etc. are in one folder. To increase structure of the repo

In [3]:
def generate_pipeline_paths(video_path, base_output="../output"):
    video_title = os.path.splitext(os.path.basename(video_path))[0]
    root_dir = os.path.join(base_output, video_title)

    return {
        "root": root_dir,
        "audio_raw": os.path.join(root_dir, "audio", "original_unprocessed.wav"),
        "audio_vocals": os.path.join(root_dir, "audio", "vocals_raw.wav"),
        "audio_vocals_48k": os.path.join(root_dir, "audio", "vocals_48k.wav"),
        "audio_vocals_denoised": os.path.join(root_dir, "audio", "vocals_denoised.wav"),
        "video_muted": os.path.join(root_dir, "video", "video_muted.mp4"),
        "video_final": os.path.join(root_dir, "video", "video_final.mp4"),
        "transcript_json": os.path.join(root_dir, "transcripts", "transcript.json"),
        "demucs_out": os.path.join(root_dir, "demucs")
    }


this is the main function to use the entire pipeline 

Von MP4 zu Audio + Background + Transcriped text
   

In [4]:
def full_video_audio_pipeline(video_path, base_output="../output", enable_diarization=True, min_speakers=1, max_speakers=10, language=None):
    """
    Complete pipeline: video → audio separation → vocal isolation → denoising → WhisperX transcription with diarization
    Skips preprocessing steps if intermediate files already exist.
    """
    from df.enhance import enhance, init_df, load_audio, save_audio

    paths = generate_pipeline_paths(video_path, base_output)
    os.makedirs(paths["root"], exist_ok=True)

    # Step 1: Separate audio and muted video (skip if files exist)
    if not os.path.exists(paths["audio_raw"]) or not os.path.exists(paths["video_muted"]):
        print("🎵 Separating audio from video...")
        separate_audio_from_video(video_path, paths["audio_raw"], paths["video_muted"])
    else:
        print("✅ Raw audio and muted video already exist, skipping separation...")

    # Step 2: Separate vocals with Demucs (skip if vocals exist)
    if not os.path.exists(paths["audio_vocals"]):
        print("🎤 Separating vocals with Demucs...")
        vocals_path, _ = separate_vocals(paths["audio_raw"], output_dir=os.path.dirname(paths["audio_vocals"]))
        shutil.move(vocals_path, paths["audio_vocals"])  # Rename into our structure
    else:
        print("✅ Vocals already separated, skipping Demucs...")

    # Step 3: Convert vocals to 48kHz WAV (skip if converted file exists)
    if not os.path.exists(paths["audio_vocals_48k"]):
        print("🔄 Converting vocals to 48kHz WAV...")
        convert_to_48k_wav(paths["audio_vocals"], paths["audio_vocals_48k"])
    else:
        print("✅ 48kHz vocals already exist, skipping conversion...")

    # Step 4: Denoise vocals (skip if denoised file exists)
    if not os.path.exists(paths["audio_vocals_denoised"]):
        print("🔧 Denoising vocals...")
        model_df, df_state, _ = init_df()
        audio, _ = load_audio(paths["audio_vocals_48k"], sr=df_state.sr())
        enhanced = enhance(model_df, df_state, audio)
        save_audio(paths["audio_vocals_denoised"], enhanced, df_state.sr())
    else:
        print("✅ Denoised vocals already exist, skipping denoising...")

    # Step 5: Transcribe with WhisperX API (always run unless transcript exists)
    # Generate transcript paths if not in the paths dict
    transcript_json_path = paths.get("transcript_json", os.path.join(paths["root"], "transcript.json"))
    transcript_srt_path = paths.get("transcript_srt", os.path.join(paths["root"], "transcript.srt"))
    
    if not os.path.exists(transcript_json_path) or not os.path.exists(transcript_srt_path):
        print("🎤 Starting transcription with WhisperX...")
        result = transcribe_with_whisperx(
            paths["audio_raw"], 
            language=language,
            enable_diarization=enable_diarization,
            min_speakers=min_speakers,
            max_speakers=max_speakers
        )
        
        if result:
            write_transcription_to_json(result, transcript_json_path)
            write_transcription_to_srt(result, transcript_srt_path)
        else:
            print("❌ Transcription failed!")
    else:
        print("✅ Transcription files already exist, skipping WhisperX...")

    print(f"🎉 Pipeline complete. All outputs saved under: {paths['root']}")
    return paths


Here is how to use this pipeline:

In [5]:
# Updated usage example with diarization options
video_path = "../input/job.mp4"
pipeline_paths = full_video_audio_pipeline(
    video_path, 
    enable_diarization=True,  # Enable speaker separation
    min_speakers=1,           # Minimum number of speakers
    max_speakers=2,           # Maximum number of speakers  
    language="en"             # Language (optional, auto-detect if None)
)


✅ Raw audio and muted video already exist, skipping separation...
✅ Vocals already separated, skipping Demucs...
✅ 48kHz vocals already exist, skipping conversion...
✅ Denoised vocals already exist, skipping denoising...
🎤 Starting transcription with WhisperX...
Transcribing audio file with WhisperX: ../output/job/audio/original_unprocessed.wav
✅ WhisperX service is healthy
📤 Sending audio to WhisperX API...
   Language: en
   Diarization: true
   Speaker range: 1-2
✅ Transcription complete. Found 0 segments.
✅ JSON transcript saved to: ../output/job/transcripts/transcript.json
✅ SRT transcript saved to: ../output/job/transcript.srt
🎉 Pipeline complete. All outputs saved under: ../output/job
